In [1]:
import chess
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, ReLU,
                                     Add, Flatten, Dense, GlobalAveragePooling2D, Reshape, Multiply)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, Callback
from tensorflow.keras.utils import Sequence
from sklearn.model_selection import train_test_split
import pandas as pd
import h5py
import json
from tqdm import tqdm
import argparse
import os
import sys

# --- GPU and Performance Optimization ---
# Enable mixed precision training for significant speedup on compatible GPUs
tf.keras.mixed_precision.set_global_policy('mixed_float16')

# Configure TensorFlow to use GPU and manage memory growth
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Running on {len(gpus)} GPU(s).")
    except RuntimeError as e:
        print(e)


# --- Enhanced Feature Engineering ---
# Total planes: 12 (pieces) + 4 (castling) + 1 (en passant) + 1 (turn)
# + 1 (halfmove) + 1 (fullmove) + 2 (king safety) = 22 planes

def gen_x_data(fen):
    """
    Converts a FEN string to a more comprehensive 8x8xN tensor representation.
    This version includes king safety features and is robust to data types and whitespace.
    """
    if not isinstance(fen, str):
        return None
    try:
        cleaned_fen = fen.strip()
        if not cleaned_fen: return None
        board = chess.Board(cleaned_fen)
    except (ValueError, AttributeError):
        return None 

    piece_planes = np.zeros((8, 8, 12), dtype=np.int8)
    piece_map = {p: i for i, p in enumerate([chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING])}
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece:
            plane_idx = piece_map[piece.piece_type] + (6 if piece.color == chess.BLACK else 0)
            row, col = divmod(square, 8)
            piece_planes[row, col, plane_idx] = 1

    castling_planes = np.zeros((8, 8, 4), dtype=np.int8)
    if board.has_kingside_castling_rights(chess.WHITE): castling_planes[:, :, 0] = 1
    if board.has_queenside_castling_rights(chess.WHITE): castling_planes[:, :, 1] = 1
    if board.has_kingside_castling_rights(chess.BLACK): castling_planes[:, :, 2] = 1
    if board.has_queenside_castling_rights(chess.BLACK): castling_planes[:, :, 3] = 1

    en_passant_plane = np.zeros((8, 8, 1), dtype=np.int8)
    if board.ep_square is not None:
        row, col = divmod(board.ep_square, 8)
        en_passant_plane[row, col, 0] = 1

    turn_plane = np.ones((8, 8, 1), dtype=np.int8) * int(board.turn)
    halfmove_plane = np.ones((8, 8, 1), dtype=np.float32) * min(board.halfmove_clock / 100.0, 1.0)
    fullmove_plane = np.ones((8, 8, 1), dtype=np.float32) * min(board.fullmove_number / 100.0, 1.0)
    
    king_safety_planes = np.zeros((8, 8, 2), dtype=np.int8)
    for square in chess.SQUARES:
        row, col = divmod(square, 8)
        if board.is_attacked_by(chess.WHITE, square): king_safety_planes[row, col, 0] = 1
        if board.is_attacked_by(chess.BLACK, square): king_safety_planes[row, col, 1] = 1

    tensor = np.concatenate([
        piece_planes, castling_planes, en_passant_plane,
        turn_plane, halfmove_plane, fullmove_plane, king_safety_planes
    ], axis=-1)
    return tensor.astype(np.float32)


# --- Data Preprocessing ---

def preprocess_and_save_data(df, output_path='preprocessed_data.h5'):
    """
    Preprocesses the FENs from the dataframe and saves them into an HDF5 file.
    Returns the determined input shape for the model.
    """
    input_shape = None
    for fen in df['fen']:
        sample_tensor = gen_x_data(fen)
        if sample_tensor is not None:
            input_shape = sample_tensor.shape
            break
    if input_shape is None:
        raise ValueError("No valid FENs found in the dataset. Please check your data.")
    
    with h5py.File(output_path, 'w') as hf:
        X_dset = hf.create_dataset('X', (len(df),) + input_shape, dtype='float32', maxshape=(None,) + input_shape)
        y_dset = hf.create_dataset('y', (len(df),), dtype='float32', maxshape=(None,))
        
        processed_count = 0
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Preprocessing FENs"):
            tensor = gen_x_data(row['fen'])
            if tensor is not None:
                X_dset[processed_count] = tensor
                y_dset[processed_count] = np.tanh(float(row['evaluation']) / 1000.0)
                processed_count += 1
        
        print(f"\nTotal valid positions processed: {processed_count}")
        if processed_count == 0:
            raise ValueError("Preprocessing resulted in zero valid positions. Please check your FEN data.")
            
        X_dset.resize((processed_count,) + input_shape)
        y_dset.resize((processed_count,))
    
    return input_shape

# --- Stable Keras Sequence Data Generator ---
class HDF5DataGenerator(Sequence):
    """
    Generates data for Keras from an HDF5 file. This is a stable alternative to tf.data.Dataset.from_generator.
    """
    def __init__(self, db_path, indices, batch_size):
        self.db_path = db_path
        self.indices = indices
        self.batch_size = batch_size

    def __len__(self):
        return int(np.floor(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        with h5py.File(self.db_path, 'r') as hf:
            batch_indices = self.indices[index*self.batch_size:(index+1)*self.batch_size]
            batch_indices.sort()
            X = hf['X'][batch_indices]
            y = hf['y'][batch_indices]
        return X, y


# --- Model Architecture (ResNet with Squeeze-and-Excite) ---

def se_block(input_tensor, ratio=16):
    """Creates a Squeeze-and-Excite block."""
    init = input_tensor
    channel_axis = -1
    filters = init.shape[channel_axis]
    se_shape = (1, 1, filters)
    se = GlobalAveragePooling2D()(init)
    se = Reshape(se_shape)(se)
    se = Dense(filters // ratio, activation='relu', kernel_initializer='he_normal', use_bias=False)(se)
    se = Dense(filters, activation='sigmoid', kernel_initializer='he_normal', use_bias=False)(se)
    x = Multiply(dtype='float32')([tf.cast(init, 'float32'), se])
    return x

def se_residual_block(x, filters):
    """A single residual block with a Squeeze-and-Excite block."""
    res = x
    x = Conv2D(filters, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(filters, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = se_block(x)
    x = Add()([x, res])
    x = ReLU()(x)
    return x

def build_model(input_shape, num_filters=128, num_blocks=8):
    """Builds the deep SE-ResNet."""
    inputs = Input(shape=input_shape)
    x = tf.cast(inputs, tf.float32)
    x = Conv2D(num_filters, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    for _ in range(num_blocks):
        x = se_residual_block(x, num_filters)
    value_head = Conv2D(1, 1, padding='same')(x)
    value_head = BatchNormalization()(value_head)
    value_head = ReLU()(value_head)
    value_head = Flatten()(value_head)
    value_head = Dense(256, activation='relu')(value_head)
    outputs = Dense(1, activation='tanh', name='evaluation', dtype='float32')(value_head)
    model = Model(inputs=inputs, outputs=outputs)
    return model

# --- Custom TQDM Callback for Training Progress ---
class TqdmProgressCallback(Callback):
    """A custom callback to display a TQDM progress bar with sample count."""
    def __init__(self, train_generator):
        super().__init__()
        self.train_generator = train_generator

    def on_epoch_begin(self, epoch, logs=None):
        self.total_samples = len(self.train_generator.indices)
        self.epoch_tqdm = tqdm(total=self.params['steps'], desc=f"Epoch {epoch + 1}/{self.params['epochs']}", unit="step", leave=False)

    def on_train_batch_end(self, batch, logs=None):
        self.epoch_tqdm.update(1)
        # batch is 0-indexed
        samples_seen = (batch + 1) * self.train_generator.batch_size
        samples_seen = min(samples_seen, self.total_samples)
        self.epoch_tqdm.set_postfix(loss=f"{logs.get('loss'):.4f}", samples=f"{samples_seen}/{self.total_samples}")

    def on_epoch_end(self, epoch, logs=None):
        self.epoch_tqdm.close()
        log_items = [f"{key}: {value:.4f}" for key, value in logs.items()]
        print(" - ".join(log_items))


# --- Main Training Script ---

def main(args):
    # 1. Load your dataset from JSON files
    print("Loading dataset from JSON files...")
    try:
        with open(args.fens_path, 'r') as f:
            fens_data = json.load(f)
            fens = fens_data.get('fens', fens_data)
        with open(args.evals_path, 'r') as f:
            evals_data = json.load(f)
            evaluations = evals_data.get('y_data', evals_data)
        df = pd.DataFrame({'fen': fens, 'evaluation': evaluations})
        print(f"Successfully loaded {len(df)} positions.")
    except FileNotFoundError as e:
        print(f"Error: Could not find a data file. {e}")
        exit()
    except Exception as e:
        print(f"An error occurred during data loading: {e}")
        exit()

    # 2. Preprocess and save data to HDF5
    HDF5_FILE = 'chess_data2.h5'
    input_shape = None
    if not os.path.exists(HDF5_FILE) or args.force_preprocess:
        print("Preprocessing data and saving to HDF5...")
        input_shape = preprocess_and_save_data(df, HDF5_FILE)
        print("Data saved to", HDF5_FILE)
    else:
        print("Found existing HDF5 file. Skipping preprocessing.")

    # 3. Create train/validation split
    with h5py.File(HDF5_FILE, 'r') as hf:
        num_samples = len(hf['X'])
        if input_shape is None:
            input_shape = hf['X'].shape[1:]
    
    print(f"Total valid samples for training: {num_samples}")
    indices = np.arange(num_samples)
    # Since the data is already shuffled, we can just split it.
    train_indices = indices[:int(num_samples * 0.85)]
    val_indices = indices[int(num_samples * 0.85):]

    # 4. Create Keras Sequence generators
    print("Creating data generators...")
    train_gen = HDF5DataGenerator(HDF5_FILE, train_indices, args.batch_size)
    val_gen = HDF5DataGenerator(HDF5_FILE, val_indices, args.batch_size)

    # 5. Build and compile the model
    print("Building model...")
    model = build_model(input_shape=input_shape, num_filters=args.filters, num_blocks=args.blocks)
    optimizer = Adam(learning_rate=args.learning_rate)
    model.compile(optimizer=optimizer, loss='mean_squared_error')
    model.summary()

    # 6. Set up callbacks
    checkpoint = ModelCheckpoint('best_chess_model.h5', monitor='val_loss', save_best_only=True, mode='min', verbose=0)
    early_stopping = EarlyStopping(monitor='val_loss', patience=5, mode='min', verbose=1)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6, mode='min', verbose=1)
    tqdm_callback = TqdmProgressCallback(train_gen)

    # 7. Train the model
    print("\nStarting training...")
    model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=args.epochs,
        callbacks=[checkpoint, early_stopping, reduce_lr, tqdm_callback],
        workers=args.workers,
        use_multiprocessing=True,
        verbose=0 
    )

    print("\nTraining finished!")
    print("The best model has been saved to 'best_chess_model.h5'")


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description="Train a chess evaluation neural network.")
    parser.add_argument('--fens_path', type=str, default='positions.json', help='Path to the JSON file containing FEN strings.')
    parser.add_argument('--evals_path', type=str, default='ydata.json', help='Path to the JSON file containing evaluations.')
    parser.add_argument('--batch_size', type=int, default=4096, help='Batch size for training. Larger is often better for GPUs.')
    parser.add_argument('--epochs', type=int, default=50, help='Maximum number of training epochs.')
    parser.add_argument('--learning_rate', type=float, default=1e-3, help='Initial learning rate for the Adam optimizer.')
    parser.add_argument('--filters', type=int, default=128, help='Number of filters in convolutional layers.')
    parser.add_argument('--blocks', type=int, default=10, help='Number of residual blocks in the model.')
    parser.add_argument('--workers', type=int, default=4, help='Number of worker processes for data generation.')
    parser.add_argument('--force_preprocess', action='store_true', help='Force preprocessing even if HDF5 file exists.')
    
    if any('ipykernel' in arg for arg in sys.argv):
         args = parser.parse_args([])
    else:
         args = parser.parse_args()

    main(args)


INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 4060 Laptop GPU, compute capability 8.9
Running on 1 GPU(s).
Loading dataset from JSON files...
Successfully loaded 1000000 positions.
Preprocessing data and saving to HDF5...


Preprocessing FENs:   2%|▏         | 24227/1000000 [22:44<15:15:46, 17.76it/s]


KeyboardInterrupt: 